## Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

    - How to create test datasets
    - How to run your RAG application on those datasets
    - How to measure your application's performance using different evaluation metrics

#### Overview

A typical RAG evaluation workflow consists of three main steps:

Creating a dataset with questions and their expected answers
Running your RAG application on those questions
Using evaluators to measure how well your application performed, looking at factors like:

    - Answer relevance
    - Answer accuracy
    - Retrieval quality
    
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

### Chatbot Evaluation

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(".env")

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [2]:
# Create datapoints
from langsmith import Client
from langsmith.utils import LangSmithConflictError

client = Client()

# Define the dataset - this is your test data container
dataset_name = "Chatbots Evaluations"
chatbot_examples = [
    {
        "inputs": {"question": "What is LangChain?"},
        "outputs": {"answer": "A framework for building LLM applications"},
    },
    {
        "inputs": {"question": "What is LangSmith?"},
        "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
    },
    {
        "inputs": {"question": "What is OpenAI?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
    {
        "inputs": {"question": "What is Google?"},
        "outputs": {"answer": "A technology company known for search"},
    },
    {
        "inputs": {"question": "What is Mistral?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
]

try:
    dataset = client.create_dataset(dataset_name, description="Chatbots Evaluations")
except LangSmithConflictError:
    dataset = client.read_dataset(dataset_name=dataset_name)

# Avoid duplicate examples when this cell is re-run.
existing_chatbot_questions = {
    ex.inputs.get("question")
    for ex in client.list_examples(dataset_id=dataset.id)
    if isinstance(ex.inputs, dict) and ex.inputs.get("question")
}
new_chatbot_examples = [
    ex for ex in chatbot_examples if ex["inputs"]["question"] not in existing_chatbot_questions
]
if new_chatbot_examples:
    client.create_examples(
        dataset_id=dataset.id,
        examples=new_chatbot_examples,
    )
else:
    print("No new chatbot examples to add.")

### Define a Metrices (LLM AS A JUDGE)

In [3]:
import anthropic
from langsmith import wrappers

anthropic_client = wrappers.wrap_anthropic(anthropic.Anthropic())
eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def chatbot_correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    predicted = (outputs or {}).get("response")
    reference = (reference_outputs or {}).get("answer")
    if not predicted or not reference:
        return False

    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference}
    You are grading the following predicted answer:
    {predicted}
    Respond with CORRECT or INCORRECT:
    Grade:
    """

    response = anthropic_client.messages.create(
        model="claude-3-5-sonnet-20241022",
        system=eval_instructions,
        max_tokens=10,
        temperature=0,
        messages=[
            {"role": "user", "content": user_content},
        ],
    ).content[0].text

    return response.strip().upper() == "CORRECT"

In [4]:
# Concisions -> check whether the actual output is less than 2x the length of the expected results

def concisions(inputs: dict, outputs: dict, reference_outputs: dict) -> int:
    predicted = (outputs or {}).get("response", "")
    reference = (reference_outputs or {}).get("answer", "")

    if not predicted or not reference:
        return 0

    return int(len(predicted) < 2 * len(reference))

## Run Evaluations

In [5]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

def my_app(question: str, model: str = "claude-3-5-sonnet-20241022", instructions: str = default_instructions) -> str:
    response = anthropic_client.messages.create(
        model=model,
        system=instructions,
        max_tokens=256,
        temperature=0,
        messages=[
            {"role": "user", "content": question},
        ],
    )
    return response.content[0].text if response.content else ""

In [6]:
# Call my_app for every datapoint

def ls_target(inputs: dict) -> dict:
    try:
        return {"response": my_app(inputs["question"])}
    except Exception as e:
        # Keep a consistent output schema so evaluators don't fail with KeyError.
        return {"response": "", "error": str(e)}

In [19]:
# Run our evaluation

experiment_results = client.evaluate(
    ls_target,   # Your AI system
    data=dataset_name,
    evaluators=[chatbot_correctness, concisions],
    experiment_prefix="claude-3-5-sonnet-20241022-chatbot",
)

View the evaluation results for experiment: 'claude-3-5-sonnet-20241022-chatbot-ce4ba380' at:
https://smith.langchain.com/o/2c0767dd-edc2-4505-b9ac-cb4b1af917b6/datasets/3d27b9e5-e3ba-4d12-be18-d36c6daeae62/compare?selectedSessions=f7e36ebe-3b1f-4f1c-b9d6-38717fbdefbe




5it [00:02,  2.20it/s]


## Evaluations for RAG

In [8]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


# List of urls to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents for the urls
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=250, chunk_overlap=0)

# split the document into chunks 
docs_split = text_splitter.split_documents(docs_list)

# Add the documents chunks to the "vector store" using Huggingface Embeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents=docs_split,
    embedding=HuggingFaceEmbeddings()
) 

# with langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=6)

USER_AGENT environment variable not set, consider setting it to identify your requests.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6862.17it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
results = retriever.invoke("What are LLM-powered autonomous agents?")
print(results)

[Document(id='2e60b183-1a5d-43cd-a11c-443a49130834', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [10]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv(".env")

#initlialize the groq llm 

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key = groq_api_key ,
    model_name ="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)

In [11]:
from langsmith import traceable

# Add decorator

@traceable()
def rag_bot(question: str) -> dict:
    # Relevant context
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.
Use the following source documents to answer the user's questions.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""

    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])

    return {"answer": ai_msg.content, "documents": docs}

In [12]:
print(rag_bot("What are LLM-powered autonomous agents?"))

{'answer': 'LLM‑powered autonomous agents are systems that use a large language model as the core “brain” to reason, plan, and act. They combine the LLM with components for planning (task decomposition and self‑reflection), memory (short‑term and long‑term storage, often via MIPS), and tool use (calling external APIs or programs). Together these parts let the agent break down complex tasks, learn from its own actions, and interact with the world autonomously.', 'documents': [Document(id='2e60b183-1a5d-43cd-a11c-443a49130834', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerf

In [13]:
from langsmith import Client
from langsmith.utils import LangSmithConflictError

client = Client()

# Define the examples for the RAG dataset
rag_examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    },
]

# Create (or read) the RAG evaluation dataset
rag_dataset_name = "RAG_TEST_EVALUATIONS"
try:
    rag_dataset = client.create_dataset(dataset_name=rag_dataset_name)
except LangSmithConflictError:
    rag_dataset = client.read_dataset(dataset_name=rag_dataset_name)

# Avoid duplicate examples when this cell is re-run.
existing_rag_questions = {
    ex.inputs.get("question")
    for ex in client.list_examples(dataset_id=rag_dataset.id)
    if isinstance(ex.inputs, dict) and ex.inputs.get("question")
}
new_rag_examples = [
    ex for ex in rag_examples if ex["inputs"]["question"] not in existing_rag_questions
]
if new_rag_examples:
    client.create_examples(
        dataset_id=rag_dataset.id,
        examples=new_rag_examples,
    )
else:
    print("No new RAG examples to add.")

## Evaluators or metrices

1 . Correctness: Response vs reference answer

- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-judge to assess answer correctness.

In [14]:
from typing_extensions import Annotated,TypedDict

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]
    
# Correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""


grader_llm = ChatGroq(
    groq_api_key = groq_api_key ,
    model_name ="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=1024
).with_structured_output(CorrectnessGrade , method="json_schema", strict = True)

def correctness(inputs: dict , outputs: dict , reference_outputs: dict)->bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system" , "content": correctness_instructions},
        {"role": "user" , "content": answers}
    ])

    return grade["correct"]


    

## Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [15]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""


relevance_llm = ChatGroq(
    groq_api_key = groq_api_key ,
    model_name ="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=1024
).with_structured_output(RelevanceGrade , method="json_schema", strict = True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

## Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [16]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. """

# Grader LLM
grounded_llm = ChatGroq(
    groq_api_key = groq_api_key ,
    model_name ="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=1024
).with_structured_output(GroundedGrade , method="json_schema", strict = True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]


## Retrieval Relevance: Retrieved docs vs input

In [17]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""


retrieval_relevance_llm = ChatGroq(
    groq_api_key = groq_api_key ,
    model_name ="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=1024
).with_structured_output(RetrievalRelevanceGrade , method="json_schema", strict = True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]



## Run the Evaluations

In [18]:
def target(inputs: dict)-> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=rag_dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, openai/gpt-oss-20b"},
)
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-6f4dc621' at:
https://smith.langchain.com/o/2c0767dd-edc2-4505-b9ac-cb4b1af917b6/datasets/cc399d64-59bc-4a05-b131-8043446af5b1/compare?selectedSessions=79d028dc-c5af-4ac1-83af-45d1821ad7b6




3it [00:48, 16.15s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are the types of biases that can arise wi...,Few‑shot prompting can introduce **selection b...,[page_content='Instruction Prompting#\nThe pur...,None,The biases that can arise with few-shot prompt...,False,False,True,False,0.825978,3d76ca3c-ebbe-4c75-b662-facd41fdee90,019da626-0ce0-7593-885b-27935b1c99f0
1,What are five types of adversarial attacks?,The five types of adversarial attacks are: \n...,[page_content='Black-box attacks assume that a...,None,Five types of adversarial attacks are (1) Toke...,True,True,True,True,0.353052,45b33c46-ee50-4750-b436-5935d5e94402,019da626-1bb4-7773-8679-3fb48285cc29
2,How does the ReAct agent use self-reflection?,ReAct agents embed self‑reflection by generati...,"[page_content='Each element is an observation,...",None,"ReAct integrates reasoning and acting, perform...",False,True,True,True,6.976738,fd0de845-ce55-466b-9923-6f88fd713465,019da626-3b50-7b60-a76b-e8f1cdfb6ca3
